In [3]:
import json, re
from collections import Counter

RANGES = {
    "khoa":  ((2506, 10788), (2506, 15787)),
    "kiet":  ((2505, 20666), (2506, 787)),
    "hoang": ((2411, 13799), (2411, 18798)),
    "tan":   ((2407, 17839), (2408, 1044)),
}

ID_RE = re.compile(r"^(\d{4})\.(\d{4,5})$")   # chỉ nhận ID dạng mới, bỏ ID cũ như hep-th/9901001

def parse_id(arxiv_id):
    m = ID_RE.match(arxiv_id)
    return (int(m.group(1)), int(m.group(2))) if m else None

def find_owner(key):
    for name, (lo, hi) in RANGES.items():
        if lo <= key <= hi:      # so sánh cặp (tháng, số thứ tự)
            return name
    return None

outs = {name: open(f"papers_{name}.jsonl", "w", encoding="utf-8") for name in RANGES}
counts = Counter()

with open("arxiv-metadata-oai-snapshot.json", "r", encoding="utf-8") as fin:
    for line in fin:
        paper = json.loads(line)
        key = parse_id(paper["id"])
        if key is None:
            continue
        owner = find_owner(key)
        if owner:
            outs[owner].write(line)
            counts[owner] += 1

for f in outs.values():
    f.close()

print(dict(counts))

{'tan': 5000, 'hoang': 5000, 'kiet': 5000, 'khoa': 5000}


In [4]:
def check(name):
    ids, ver_counts = [], Counter()
    with open(f"papers_{name}.jsonl", encoding="utf-8") as f:
        for line in f:
            p = json.loads(line)
            ids.append(p["id"])
            ver_counts[len(p["versions"])] += 1

    by_month = {}
    for i in ids:
        ym, n = parse_id(i)
        by_month.setdefault(ym, []).append(n)

    print(f"=== {name} ===")
    print("Số bài:", len(ids), "| Số ID khác nhau:", len(set(ids)))
    for ym, ns in sorted(by_month.items()):
        ns.sort()
        gaps = (ns[-1] - ns[0] + 1) - len(ns)
        print(f"  Tháng {ym}: từ {ns[0]} đến {ns[-1]}, {len(ns)} bài, ID thiếu trong khoảng: {gaps}")
    total_versions = sum(k * v for k, v in ver_counts.items())
    print("Phân bố (số phiên bản, số bài):", sorted(ver_counts.items()))
    print("Tổng số phiên bản:", total_versions)
    print()

for name in RANGES:
    check(name)

=== khoa ===
Số bài: 5000 | Số ID khác nhau: 5000
  Tháng 2506: từ 10788 đến 15787, 5000 bài, ID thiếu trong khoảng: 0
Phân bố (số phiên bản, số bài): [(1, 2936), (2, 1427), (3, 442), (4, 131), (5, 43), (6, 14), (7, 1), (8, 3), (10, 1), (16, 1), (17, 1)]
Tổng số phiên bản: 8013

=== kiet ===
Số bài: 5000 | Số ID khác nhau: 5000
  Tháng 2505: từ 20666 đến 24878, 4213 bài, ID thiếu trong khoảng: 0
  Tháng 2506: từ 1 đến 787, 787 bài, ID thiếu trong khoảng: 0
Phân bố (số phiên bản, số bài): [(1, 2798), (2, 1478), (3, 500), (4, 150), (5, 56), (6, 12), (7, 2), (8, 4)]
Tổng số phiên bản: 8252

=== hoang ===
Số bài: 5000 | Số ID khác nhau: 5000
  Tháng 2411: từ 13799 đến 18798, 5000 bài, ID thiếu trong khoảng: 0
Phân bố (số phiên bản, số bài): [(1, 2773), (2, 1472), (3, 485), (4, 192), (5, 51), (6, 14), (7, 6), (8, 4), (9, 1), (12, 1), (44, 1)]
Tổng số phiên bản: 8418

=== tan ===
Số bài: 5000 | Số ID khác nhau: 5000
  Tháng 2407: từ 17839 đến 21794, 3956 bài, ID thiếu trong khoảng: 0
  Tháng